# Fase 1 · extracción
Modelo doble determinista. LCEL, @chain, invoke, batch y stream.

In [1]:
from pathlib import Path
import os
if Path.cwd().name == 'notebooks': os.chdir('..')

In [2]:
from langchain_core.runnables import RunnableLambda
from pedro.chains import crear_lcel, crear_chain
import json
modelo = RunnableLambda(lambda prompt: json.dumps({'intencion':'comparar',
    'solicitud':{'producto_id':'PEC-01-0001','cantidad':2,'presupuesto_centavos':100000,
    'plazo_max_dias':7},'faltantes':[]}, ensure_ascii=False))
lcel, decorada = crear_lcel(modelo), crear_chain(modelo)
entrada = {'texto':'Compara dos unidades de PEC-01-0001 hasta R$ 1.000 en 7 días'}
print(lcel.invoke(entrada).model_dump_json())
print(decorada.invoke(entrada).model_dump_json())
print([x.model_dump() for x in lcel.batch([entrada, entrada])])
print(list(modelo.stream('etapa del modelo')))
print(lcel.get_graph().draw_mermaid())

{"intencion":"comparar","solicitud":{"producto_id":"PEC-01-0001","producto":null,"cantidad":2,"presupuesto_centavos":100000,"plazo_max_dias":7,"cep":null},"faltantes":[]}
{"intencion":"comparar","solicitud":{"producto_id":"PEC-01-0001","producto":null,"cantidad":2,"presupuesto_centavos":100000,"plazo_max_dias":7,"cep":null},"faltantes":[]}
[{'intencion': 'comparar', 'solicitud': {'producto_id': 'PEC-01-0001', 'producto': None, 'cantidad': 2, 'presupuesto_centavos': 100000, 'plazo_max_dias': 7, 'cep': None}, 'faltantes': []}, {'intencion': 'comparar', 'solicitud': {'producto_id': 'PEC-01-0001', 'producto': None, 'cantidad': 2, 'presupuesto_centavos': 100000, 'plazo_max_dias': 7, 'cep': None}, 'faltantes': []}]
['{"intencion": "comparar", "solicitud": {"producto_id": "PEC-01-0001", "cantidad": 2, "presupuesto_centavos": 100000, "plazo_max_dias": 7}, "faltantes": []}']
---
config:
  flowchart:
    curve: linear
---
graph TD;
	PromptInput([PromptInput]):::first
	ChatPromptTemplate(ChatProm

LCEL expone etapas declarativas. @chain envuelve la misma extracción. Los faltantes se recalculan tras el parser.

## R01 · anatomía del agente
La anatomía se lee de los objetos reales del agente final (`pedro.anatomia`), no se escribe a mano. No se llama a ningún modelo: el executor se construye con un modelo de relleno que nunca se invoca.

In [3]:
import json
from pedro.catalog import crear_escenario
from pedro.anatomia import describir_anatomia
escenario = crear_escenario(Path('storage/notebooks/catalogos'))
memoria = Path('storage/notebooks/r01.sqlite3')
anatomia = describir_anatomia(escenario, memoria)
for clave in ('motor_cognitivo', 'orquestador', 'system_message', 'memoria',
              'base_conocimiento', 'puerta_publicacion'):
    print(clave, json.dumps(anatomia[clave], ensure_ascii=False))
for h in anatomia['herramientas']:
    print('herramienta', h['nombre'], '|', h['papel'], '|', h['logica'], '|', h['argumentos'])

motor_cognitivo {"clase": "ChatOpenAI", "modelo": "gpt-4.1-mini", "disponible": false}
orquestador {"agente": "create_openai_tools_agent", "ejecutor": "AgentExecutor", "max_iteraciones": 6, "historial": "RunnableWithMessageHistory"}
system_message {"ptcf": {"Persona": true, "Tarea": true, "Contexto": true, "Formato": true}, "variables": ["agent_scratchpad", "input"], "variables_fijadas": ["chat_history", "dataset_id", "ejemplos", "formato"], "mensajes": ["SystemMessagePromptTemplate", "MessagesPlaceholder", "HumanMessagePromptTemplate", "MessagesPlaceholder"]}
memoria {"corto_plazo": "ventana de 12 mensajes + borrador por sesión", "largo_plazo": "preferencias explícitas + episodios vectoriales por usuario", "modelo_embeddings": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"}
base_conocimiento {"dataset_id": "e42c75fb-608f-4383-a07e-068a324e257f", "piezas": 2640, "fragmentos_politicas": 75, "origen_politicas": "ficticio_demo"}
puerta_publicacion {"verificacion": "exacta (

## R01 · agente híbrido: capa reactiva y capa deliberativa
Tres turnos sobre el mismo flujo final. **Doble de prueba:** el modelo es un guion determinista (no es OpenAI) y ViaCEP está simulado. Lo real es el flujo: las reglas, el `AgentExecutor`, la herramienta ejecutada y la verificación antes de publicar. Los dos primeros turnos los resuelve una regla sin llamar al modelo; el tercero pasa por el modelo, que recibe las seis herramientas y solicita una.

In [4]:
from langchain_core.messages import AIMessage
from langchain_core.runnables import RunnableLambda
from pedro.agent import Traza, crear_flujo
from pedro.memory import nueva_sesion
sesion = nueva_sesion(memoria, 'alumna', escenario.dataset_id)
final = json.dumps({'estado': 'cep', 'dataset_id': escenario.dataset_id,
    'solicitud': {'cep': '01001000'}, 'cidade': 'Cidade Simulada', 'uf': 'SP',
    'modo_cep': 'simulado', 'evidencias': [{'tipo': 'viacep',
    'fuente_id': 'ViaCEP:01001000', 'texto': 'Cidade Simulada/SP'}]})
guion = [AIMessage(content='', tool_calls=[{'name': 'consultar_cep',
                                             'args': {'cep': '01001000'}, 'id': 'c1'}]),
         AIMessage(content=final)]
llamadas = []
def modelo_doble(prompt, **kwargs):
    llamadas.append(sorted(t['function']['name'] for t in kwargs.get('tools', [])))
    return guion[len(llamadas) - 1]
traza = Traza()
flujo = crear_flujo(escenario, memoria, llm=RunnableLambda(modelo_doble),
                    cep_simulado=True, traza=traza)
config = {'configurable': {'user_id': 'alumna', 'session_id': sesion}}
for pregunta in ('lista piezas', 'Compara PEC-01-0001',
                 '¿A qué ciudad corresponde el CEP 01001000?'):
    antes = (len(llamadas), len(traza.eventos))
    respuesta = flujo.invoke({'input': pregunta}, config=config).content
    usadas = [e['herramienta'] for e in traza.eventos[antes[1]:] if e['evento'] == 'tool_start']
    n = len(llamadas) - antes[0]
    print('turno:', pregunta)
    print('  capa:', 'deliberativa' if n else 'reactiva', '| llamadas al modelo:', n,
          '| herramientas ejecutadas:', usadas)
    print('  respuesta:', respuesta[:100].replace('\n', ' '))
print('herramientas ofrecidas al modelo:', llamadas[0])
assert len(llamadas) == 2 and [e['herramienta'] for e in traza.eventos
                               if e['evento'] == 'tool_start'] == ['consultar_cep']

turno: lista piezas
  capa: reactiva | llamadas al modelo: 0 | herramientas ejecutadas: []
  respuesta: Piezas del escenario (2640; página 1): - PEC-01-0001 · NEVE-COMP-0001 · Compresor para Neveras NEVE-


turno: Compara PEC-01-0001
  capa: reactiva | llamadas al modelo: 0 | herramientas ejecutadas: []
  respuesta: ¿Cuántas unidades de PEC-01-0001 necesitas? Puedo comparar sin presupuesto o plazo máximo si no tien
turno: ¿A qué ciudad corresponde el CEP 01001000?
  capa: deliberativa | llamadas al modelo: 2 | herramientas ejecutadas: ['consultar_cep']
  respuesta: CEP 01001000: Cidade Simulada/SP (ViaCEP, modo simulado).
herramientas ofrecidas al modelo: ['buscar_productos', 'comparar_ofertas', 'consultar_catalogo', 'consultar_cep', 'consultar_politicas', 'guardar_preferencia']


/tmp/ipykernel_2788472/4119760277.py:18: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  flujo = crear_flujo(escenario, memoria, llm=RunnableLambda(modelo_doble),


Lectura: «lista piezas» y «Compara PEC-01-0001» se resuelven por regla (0 llamadas al modelo). La pregunta de CEP no tiene regla: el modelo elige `consultar_cep` entre seis herramientas y la respuesta solo se publica tras coincidir con la observación de la herramienta. Con un modelo real la elección la hace el LLM; esa ejecución remota corresponde a R26.